# Forecast daily mine energy consumption
Run the first notebook in history mode first. Upload historical_plans.csv and future_plans.csv into the same volume. Run the package cell, then the remaining cells in order.

In [ ]:
%pip install scikit-learn==1.8.0

In [ ]:
import csv
import re
from pathlib import Path

CATALOG = "dbw_sales_iman01"  # Same value as notebook 01.
SCHEMA = "iman_energy_etl_lab"
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
prefix = f"`{CATALOG}`.`{SCHEMA}`"
volume = Path(f"/Volumes/{CATALOG}/{SCHEMA}/incoming")

def plans(filename):
    with (volume / filename).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

actual = {(str(r.date), r.site): r.total_kwh
          for r in spark.table(f"{prefix}.gold_daily_energy_cost").take(113)}
past_plans = plans("historical_plans.csv")
keys = {(r["date"], r["site"]) for r in past_plans}
if len(actual) != 112 or len(past_plans) != 112 or keys != actual.keys():
    raise ValueError("Run notebook 01 with USE_HISTORY=True before training")
history = [dict(r, total_kwh=actual[r["date"], r["site"]]) for r in past_plans]
future = plans("future_plans.csv")


## Train and evaluate
Use October for training and later observed days for testing. Compare with each site's training mean; the target is daily kWh. Future rows have no actual kWh.

In [ ]:
import math
from statistics import mean

from sklearn.ensemble import RandomForestRegressor

FEATURES = ("site_north", "planned_hours", "load_index", "forecast_temp_c")
CUTOFF = "2026-10-31"

def features(row):
    if row["site"] not in {"NORTH", "SOUTH"}:
        raise ValueError("Unknown site")
    return [int(row["site"] == "NORTH"),
            *(float(row[key]) for key in FEATURES[1:])]

def fit_and_score(history, future):
    train = [r for r in history if r["date"] <= CUTOFF]
    test = [r for r in history if r["date"] > CUTOFF]
    if not train or not test or not future:
        raise ValueError("Provide both historical periods and at least one future plan")
    if any(r["date"] <= max(x["date"] for x in history) for r in future):
        raise ValueError("Forecast dates must be later than historical dates")
    averages = {site: mean(r["total_kwh"] for r in train if r["site"] == site)
                for site in {r["site"] for r in train}}
    model = RandomForestRegressor(n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=1)
    model.fit([features(r) for r in train], [r["total_kwh"] for r in train])
    mae = mean(abs(r["total_kwh"] - p) for r, p in zip(test, model.predict([features(r) for r in test])))
    baseline = mean(abs(r["total_kwh"] - averages[r["site"]]) for r in test)
    prediction = [(r["date"], r["site"], round(float(p), 2))
                  for r, p in zip(future, model.predict([features(r) for r in future]))]
    assert all(math.isfinite(row[2]) for row in prediction)
    return dict(train=len(train), test=len(test), baseline_mae=baseline, model_mae=mae), prediction


In [ ]:
metrics, forecasts = fit_and_score(history, future)
print(metrics)
assert (metrics["train"], metrics["test"], len(forecasts)) == (62, 50, 10)


In [ ]:
from pyspark.sql import functions as F

forecast_table = f"{prefix}.gold_energy_forecast"
frame = spark.createDataFrame(forecasts, "date STRING, site STRING, predicted_kwh DOUBLE")
frame = frame.withColumn("date", F.to_date("date"))
frame.write.format("delta").mode("overwrite").saveAsTable(forecast_table)
display(spark.table(forecast_table).orderBy("date", "site"))
